In [0]:
#1 Load Gold Dataset
df = spark.read.table(
"workspace.ecommerce_lakehouse.gold_ml_training_dataset"
)

display(df)


In [0]:
# Convert to Pandas for training (small dataset):
pdf = df.toPandas()

In [0]:
#2 Feature Selection.Define ML features. 
feature_cols = [

"total_events",
"views",
"add_to_cart",
"purchases",
"avg_price",

"frequency",
"monetary_value",
"recency_days"

]

X = pdf[feature_cols].astype("float64")
y_purchase = pdf["label_purchase"]
y_fraud = pdf["label_fraud"]

In [0]:
#3 Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
X, y_purchase, test_size=0.2, random_state=42
)

In [0]:
#Check Your Username Path
print(dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get())

In [0]:
#4 Enable MLflow Tracking.Databricks automatically integrates MLflow.
# COMMAND ----------

import mlflow

mlflow.set_experiment(
"/Users/subhankarmaitra30@gmail.com/ecommerce_lakehouse_experiments"
)

In [0]:
#5 Train Model (Random Forest)
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from mlflow.models.signature import infer_signature

with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    # infer model signature
    signature = infer_signature(X_train, preds)

    mlflow.log_metric("accuracy", accuracy)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="purchase_model",
        signature=signature,
        input_example=X_train.iloc[:5],
        registered_model_name="ecommerce_purchase_model"
    )
   
    print("Accuracy:", accuracy)

In [0]:
#6 Train Fraud Detection Model
# Train/test split for fraud model
X_train, X_test, y_train, y_test = train_test_split(
X, y_fraud, test_size=0.2, random_state=42
)

with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    # infer model signature
    signature = infer_signature(X_train, preds)

    mlflow.log_metric("fraud_accuracy", accuracy)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="fraud_model",
        signature=signature,
        input_example=X_train.iloc[:5],
        registered_model_name="ecommerce_fraud_model"
    )

    print("Fraud Accuracy:", accuracy)